# Nepal Compound Hot–Dry (CHD) Events — Research Workflow

**Companion notebook** for the study *Increasing Compound Hot–Dry Events in Nepal*.

Cell order = concept-note phases: **0** validation gate → **1** auth & asset check → **2** map fix (no OSM) → **3** dataset sanity → **4** cube export → **5** climatology (±15-day calendar-day thresholds) → **6** 2024 pilot → **7** historical loop 1981–2025 → **8** trends (MK/Sen/FDR) → **9** MODIS vegetation → **10** CMIP6.

> Execution status: Cell 0 was executed in the authoring environment; Cells 1–10 require your Google account (Earth Engine) and Drive and carry `[run in Colab]` markers. No results in this notebook are research findings until the marked cells have run and passed their QA gates.


In [1]:
# CELL 0 -- Validation gate (Phase C): synthetic episode tests
# These tests MUST pass before any real-data pilot is interpreted.
# Captured output below was produced by executing the suite.
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/', '-v', '-q'],
                   capture_output=True, text=True)
print(r.stdout[-3500:])
assert r.returncode == 0, 'VALIDATION GATE FAILED'


============================= test session starts ==============================
platform linux -- Python 3.12.16, pytest-9.0.2, pluggy-1.6.0
rootdir: /home/z
configfile: pyproject.toml
plugins: Faker-40.1.2, metadata-3.1.1, asyncio-1.3.0, ddtrace-4.2.2, cov-7.0.0, json-report-1.5.0, anyio-4.13.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 18 items

tests/test_events.py ............                                        [ 66%]
tests/test_trends_spi.py ......                                          [100%]

============================== 18 passed in 1.27s ==============================


## Cell 1 — Authenticate & verify the district asset `[run in Colab]`
Sets the Cloud project `mygeepr`, prints the district feature count, field names and province list. Do not proceed unless it reports **77 features** matching the uploaded `District.shp`.


In [ ]:
# ============================================================================
# CELL 1 -- Authenticate, set the Cloud project, and VERIFY the district asset
# ============================================================================
# What it does (concept note Phase A):
#   1. Authenticates Google Earth Engine from Colab.
#   2. Initializes with the project `mygeepr` (change if the console shows
#      a different enabled project).
#   3. Prints the feature count, property names and one sample feature of the
#      Nepal district asset -- we NEVER assume 77 districts without checking.
#
# [NOT executed in this sandbox: requires your Google account. Run in Colab.]
# ============================================================================

import ee

ee.Authenticate()                       # opens the OAuth flow in Colab
ee.Initialize(project="mygeepr")        # confirm this is the right project

nepdist = ee.FeatureCollection("projects/mygeepr/assets/Nepaldist")
nepal = nepdist.geometry()

# --- asset verification -----------------------------------------------------
n_features = nepdist.size().getInfo()
first = nepdist.first().getInfo()
props = first["properties"]
print("District features:", n_features)
print("Property names:", sorted(props.keys()))
print("Sample district:", props.get("DISTRICT", props.get("District")),
      "| province:", props.get("PR_NAME", props.get("Province")))
print("Geometry type:", first["geometry"]["type"])
print("Province names:", sorted(
    nepdist.aggregate_array("PR_NAME").distinct().getInfo()))
# Expected (matches the uploaded District.shp): 77 features; fields
# OBJECTID, PROVINCE, PR_NAME, DISTRICT, geometry; polygon/multipolygon.
# The uploaded shapefile carries no .prj: coordinates are EPSG:4326 degrees.
# If the count differs from 77, STOP and reconcile the asset first.


## Cell 2 — THE MAP FIX (no OpenStreetMap, no 403) `[run in Colab]`
The 403 came from OSM's tile policy, not from Earth Engine. This cell builds the map with `tiles=None`, serves EE layers through `getMapId()`, and **proves** tile generation by probing one tile URL with `requests` (HTTP 200 = working). Cell 2b (inside the same cell) is a matplotlib fallback that always renders.


In [ ]:
# ============================================================================
# CELL 2 (THE FIX) -- Nepal districts map WITHOUT the blocked OSM basemap
# ============================================================================
# Diagnosis of the 403 error
# ---------------------------
# The 403 came from OpenStreetMap's tile servers (folium's default
# tiles="OpenStreetMap"), which blocks automated/Colab traffic by usage
# policy. It is NOT an Earth Engine problem. Three things fix it:
#   (a) create the folium map with tiles=None  -> no third-party tiles at all;
#   (b) render EE layers through ee.Image.getMapId() so Google serves the
#       EE tiles (authorization rides on your ee.Initialize session);
#   (c) verify tiles are REALLY being produced by requesting one tile URL
#       with `requests` and printing the HTTP status (200 = working).
# If (c) returns 403/404, the problem is EE-side (auth/project) -- see the
# diagnostics at the bottom. A matplotlib fallback (2b) guarantees you can
# always see the boundaries, independent of any browser/leaflet behaviour.
#
# [NOT executed in this sandbox: run in Colab after Cell 1.]
# ============================================================================

import requests

import ee
import folium
from IPython.display import display

nepdist = ee.FeatureCollection("projects/mygeepr/assets/Nepaldist")
nepal = nepdist.geometry()


def add_ee_layer(map_object, image, vis_params, name, show=True):
    """Add an EE Image to a folium map using EE's own tile service."""
    map_id = ee.Image(image).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id["tile_fetcher"].url_format,
        attr="Google Earth Engine",
        name=name,
        overlay=True,
        control=True,
        show=show,
    ).add_to(map_object)


# ---- (a) map with NO basemap tiles -----------------------------------------
m = folium.Map(location=[28.3, 84.1], zoom_start=7, tiles=None,
               control_scale=True)

# district boundaries painted as a raster layer (fill + outline)
district_image = nepdist.style(color="222222", fillColor="E8F1E8", width=1.2)
add_ee_layer(m, district_image, {}, "Nepal Districts")

# ---- (c) PROVE the tile layer is real --------------------------------------
map_id = ee.Image(district_image).getMapId({})
probe = requests.get(map_id["tile_fetcher"].url_format +
                     "/8/199/120.png", timeout=30)   # z/x/y covering Nepal
print("EE tile probe HTTP status:", probe.status_code,
      "(200 = tiles are being generated; 403/404 = EE-side problem)")

m.fit_bounds([[26.3, 80.0], [30.5, 88.3]])
folium.LayerControl(collapsed=False).add_to(m)
display(m)

# ----------------------------------------------------------------------------
# CELL 2b -- Fallback static map (matplotlib): works even if folium/leaflet
# misbehaves in the browser. Downloads district polygons once as GeoJSON.
# ----------------------------------------------------------------------------
import io

import geopandas as gpd
import matplotlib.pyplot as plt

fc = nepdist.getInfo()                     # 77 features, small payload
gdf = gpd.GeoDataFrame.from_features(fc["features"])
fig, ax = plt.subplots(figsize=(6, 7), constrained_layout=True)
gdf.plot(ax=ax, facecolor="#E8F1E8", edgecolor="#222222", linewidth=0.6)
ax.set_title("Nepal districts (EE asset verification)", fontsize=11)
ax.set_xlabel("Longitude (deg E)"); ax.set_ylabel("Latitude (deg N)")
plt.show()
print("Fallback map rendered from", len(gdf), "districts")


## Cell 3 — Dataset sanity (CHIRPS + ERA5-Land) `[run in Colab]`
Confirms band names, unit conversion and national 2024 statistics before any threshold logic runs.


In [ ]:
# ============================================================================
# CELL 3 -- Load CHIRPS + ERA5-Land and PROVE the data are real (Phase A)
# ============================================================================
# Verifies collection IDs, band names, units, coverage, and produces national
# sanity statistics for the pilot year before any threshold logic runs.
# [NOT executed in this sandbox: run in Colab after Cells 1-2.]
# ============================================================================

import calendar

import ee

ee.Initialize(project="mygeepr")

chirps = (ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
          .select("precipitation"))                 # mm/day, 0.05 deg, >=1981
era5 = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
        .select("temperature_2m_max"))              # KELVIN, ~11.1 km, >=1950

nepdist = ee.FeatureCollection("projects/mygeepr/assets/Nepaldist")
nepal = nepdist.geometry()

Y = 2024                                           # pilot year
start, end = f"{Y}-01-01", f"{Y}-12-31"

chirps_y = chirps.filterDate(start, end)
era5_y = (era5.filterDate(start, end)
          .map(lambda im: im.subtract(273.15)       # K -> degC
               .copyProperties(im, ["system:time_start"])))

# ---- coverage checks --------------------------------------------------------
n_chirps = chirps_y.size().getInfo()                # expect 366 (leap year)
n_era5 = era5_y.size().getInfo()
print(f"{Y}: CHIRPS images = {n_chirps}, ERA5-Land images = {n_era5}")

# ---- national sanity statistics (reduce over Nepal) -------------------------
mean_pr = (chirps_y.mean().reduceRegion(
    reducer=ee.Reducer.mean(), geometry=nepal, scale=11132,
    maxPixels=1e13).get("precipitation").getInfo())
tmax_stats = (era5_y.mean().reduceRegion(
    reducer=ee.Reducer.mean(), geometry=nepal, scale=11132,
    maxPixels=1e13).get("temperature_2m_max").getInfo())
print(f"National mean daily precip {Y}: {mean_pr:.2f} mm/day")
print(f"National mean daily Tmax {Y}: {tmax_stats:.1f} degC")

# Interpretation: precip ~3-6 mm/day over Nepal in a normal year (monsoon
# dominated); Tmax means ~15-25 degC. Wildly different values (e.g. ~290 for
# temperature) mean the Kelvin->Celsius conversion was skipped.


## Cell 4 — Export daily climate cubes 1981–2025 `[run in Colab]`
One multi-band GeoTIFF per variable per year (~6 MB each) on the 11.1-km ERA5-Land grid; CHIRPS aggregated with area-true mean. All further science runs locally on these cubes — reproducible and quota-safe.


In [ ]:
# ============================================================================
# CELL 4 -- Export daily climate CUBES 1981-2025 (the architecture change)
# ============================================================================
# WHY CUBES INSTEAD OF PER-DAY GEE CLOSURES
#   The original draft computed rolling sums and percentile closures inside
#   GEE for every day of 45 years -- a graph far beyond EE's limits. Instead
#   we export each year's daily Tmax/precipitation as ONE small multi-band
#   GeoTIFF (~6 MB: 366 bands x 46 rows x 85 cols), then run thresholds,
#   classification and episode detection locally (Cells 5-7). This is exact
#   (no client-side approximation), reproducible, and quota-friendly.
#
# GRID: EPSG:4326, 11132 m (~0.1 deg) = ERA5-Land native. CHIRPS (0.05 deg)
#   is aggregated with mean reduceResolution (conservative, area-true).
#   Never interpret the finer CHIRPS resolution as the analysis resolution.
# [NOT executed in this sandbox: run in Colab; exports go to your Drive.]
# ============================================================================

import time

import ee

ee.Initialize(project="mygeepr")

nepdist = ee.FeatureCollection("projects/mygeepr/assets/Nepaldist")
nepal = nepdist.geometry()

chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select("precipitation")
era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").select(
    "temperature_2m_max")

SCALE = 11132
CRS = "EPSG:4326"
YEARS = range(1981, 2026)          # 1981-2025 inclusive

# Nepal bbox with small padding (grid cells must cover edge districts)
bbox = ee.Geometry.Rectangle([79.9, 26.3, 88.3, 30.5], proj=CRS, geodesic=False)


def year_cube(imgcol, year, band, is_chirps):
    """One multi-band GeoTIFF per year: bands named YYYYMMDD."""
    ic = (imgcol.filterDate(f"{year}-01-01", f"{year + 1}-01-01")
          .filterBounds(bbox))
    if is_chirps:
        # aggregate 0.05-deg CHIRPS onto the 11.1-km grid (area-true mean)
        ic = ic.map(lambda im: im.reduceResolution(
            reducer=ee.Reducer.mean(), grid={"crs": CRS, "scale": SCALE})
            .copyProperties(im, ["system:time_start"]))
    else:
        ic = ic.map(lambda im: im.subtract(273.15)   # Kelvin -> degC
                    .copyProperties(im, ["system:time_start"]))
    img = ic.toBands().renameBands(
        lambda b: ee.String(b).replace("-", "").slice(0, 8))  # YYYYMMDD
    return img


tasks = []
for y in YEARS:
    for var, imgcol, band, is_chirps in (
            ("tmax", era5, "temperature_2m_max", False),
            ("pr", chirps, "precipitation", True)):
        img = year_cube(imgcol, y, band, is_chirps)
        t = ee.batch.Export.image.toDrive(
            image=img.clip(bbox),
            description=f"CHD_{var}_{y}",
            folder="nepal_chd_cubes",
            fileNamePrefix=f"chd_{var}_{y}",
            region=bbox, crs=CRS, scale=SCALE,
            maxPixels=1e13, formatOptions={"cloudOptimized": True})
        t.start()
        tasks.append((t.id, f"chd_{var}_{y}"))

print(f"Started {len(tasks)} export tasks (2 vars x 45 years).")
time.sleep(20)

# ---- task monitor: re-run this snippet until every task is COMPLETED --------
import collections

status = collections.Counter()
for tid, name in tasks:
    s = ee.data.getTaskStatus(tid)[0].get("state", "?")
    status[s] += 1
print(dict(status))
# Expected total runtime: ~1-2 h for all 90 tasks on a free account.
# After completion, download the folder nepal_chd_cubes to /content/chd_data
# (or mount Drive and read directly -- Cell 5 accepts either).


## Cell 5 — Climatology 1991–2020 (Phase B upgrade) `[run in Colab]`
Pixel-specific calendar-day P90 (heat) and P20 of the rolling 30-day precipitation (dry) on a ±15-day moving window; validators run as a hard gate; severity standardization fields saved.


In [ ]:
# ============================================================================
# CELL 5 -- Read cubes, VALIDATE them, build the 1991-2020 climatology (Phase B)
# ============================================================================
# Replaces the simplified monthly thresholds of the pilot with pixel-specific
# calendar-day percentiles over a +/-15-day moving window (concept note 6.1-
# 6.2, config.HEAT_PERCENTILE=90, DRY_PERCENTILE=20, CLIM_WINDOW_DAYS=15).
# Leap days are folded onto Feb 28 so the climatology has exactly 365 slots.
# Every cube passes the io_validators gate BEFORE analysis (Phase A rule).
# [NOT executed in this sandbox: run in Colab after the exports finish.]
# ============================================================================

import sys
from pathlib import Path

import numpy as np
import rasterio

sys.path.insert(0, "/content/chd_pipeline")   # where this repo is cloned
from core import io_validators
from core.events import (calendar_thresholds, dayofyear_calendar,
                         rolling_sum)
from local_analysis.run_events import standardization_climatology

import config as C

DATA = Path("/content/chd_data")             # chd_tmax_*.tif, chd_pr_*.tif
YEARS = list(range(1981, 2026))
BASE = (1991, 2020)


def read_year(var, year, data_dir=DATA):
    with rasterio.open(data_dir / f"chd_{var}_{year}.tif") as src:
        a = src.read()                        # (366, ny, nx) or (365, ...)
        prof = src.profile
        dates = np.array([np.datetime64(f"{year}-01-01") + np.timedelta64(i, "D")
                          for i in range(a.shape[0])])
        # band names are YYYYMMDD; assert they match the assumed order
        names = src.descriptions
        expected = [f"{year}{m:02d}{d:02d}" for m, d in
                    ((1, 1), (7, 1), (12, 31))]
        got = [names[0][:8], names[len(names) // 2][:8], names[-1][:8]]
        assert got == expected, f"band order mismatch {year} {var}: {got}"
    return a.astype(np.float32), dates, prof


# ---- read all years ----------------------------------------------------------
tmax_cube, dates_cube = [], []
for y in YEARS:
    a, d, _ = read_year("tmax", y)
    tmax_cube.append(a); dates_cube.append(d)
tmax = np.concatenate(tmax_cube); dates = np.concatenate(dates_cube)

# 1981 lead-in: CHIRPS starts 1981-01-01, so the first 29 days have no full
# 30-day window -> rolling_sum returns NaN there (documented exclusion).
pr_cube, pr_dates = [], []
for y in YEARS:
    a, d, _ = read_year("pr", y)
    pr_cube.append(a); pr_dates.append(d)
precip = np.concatenate(pr_cube)

ny, nx = tmax.shape[1:]
lat = np.arange(30.475, 26.275, -0.101)[:ny]   # top->bottom, ~0.1 deg
lon = np.arange(79.975, 88.475, 0.101)[:nx]
print("Cube shape:", tmax.shape, precip.shape)

# ---- validators (hard gate) --------------------------------------------------
ctx = {"dates": dates, "year": 1991, "tmax_c": tmax, "precip": precip,
       "lat": lat, "lon": lon}
ok = io_validators.run_all_checks(ctx)
assert ok, "Cube validation FAILED -- fix inputs before analysis."

# ---- calendar-day thresholds (the Phase B upgrade) --------------------------
base_mask = ((dates >= np.datetime64(f"{BASE[0]}-01-01")) &
             (dates <= np.datetime64(f"{BASE[1]}-12-31")))

thr_t = calendar_thresholds(tmax[base_mask], dates[base_mask],
                            C.HEAT_PERCENTILE, C.CLIM_WINDOW_DAYS)  # (365,H,W)
p30 = rolling_sum(precip, C.ROLLING_DAYS)
thr_p = calendar_thresholds(p30[base_mask & np.isfinite(p30)],
                            dates[base_mask & np.isfinite(p30)],
                            C.DRY_PERCENTILE, C.CLIM_WINDOW_DAYS)

clim_t = standardization_climatology(tmax, dates, base_mask,
                                     C.CLIM_WINDOW_DAYS)   # ZT fields
clim_p = standardization_climatology(p30, dates, base_mask,
                                     C.CLIM_WINDOW_DAYS)   # ZP fields

np.savez_compressed("/content/chd_outputs/climatology.npz",
                    thr_t=thr_t.astype(np.float32),
                    thr_p=thr_p.astype(np.float32),
                    clim_t_mean=clim_t["mean"].astype(np.float32),
                    clim_t_sd=clim_t["sd"].astype(np.float32),
                    clim_p_mean=clim_p["mean"].astype(np.float32),
                    clim_p_sd=clim_p["sd"].astype(np.float32),
                    lat=lat, lon=lon)
print("Climatology saved: thresholds (365,%d,%d) + Z-standardization fields"
      % thr_t.shape[1:])

# Quick climatological plausibility print (NOT results -- just QA):
july = 196                                                    # ~15 Jul
print("Sample threshold, ~15 July, grid centre:",
      float(thr_t[july, ny // 2, nx // 2]), "degC (expect ~24-32)",
      float(thr_p[july, ny // 2, nx // 2]), "mm/30d (expect monsoon-high)")


## Cell 6 — 2024 pilot: detection, maps, district tables `[run in Colab]`
Runs the validated detector for the pilot year and overlays the result grids on the boundary map (no OSM). Pilot results are exploratory until Cells 5–7 have run end-to-end.


In [ ]:
# ============================================================================
# CELL 6 -- 2024 PILOT: event detection, maps, district tables (Phase C/A)
# ============================================================================
# Runs the validated local detector on the pilot year, shows the 2024 maps in
# Colab (no OSM basemap), and exports the district CSV. The synthetic episode
# unit tests (tests/test_events.py) must pass before this cell -- they are
# executed in Cell 0 and reused here as the acceptance gate.
# [NOT executed in this sandbox: run in Colab after Cell 5.]
# ============================================================================

import sys
from pathlib import Path

import folium
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

sys.path.insert(0, "/content/chd_pipeline")
import config as C
from local_analysis.run_events import detect_for_years

OUT = Path("/content/chd_outputs")
clim = np.load(OUT / "climatology.npz")

# --- minimal cube access for the pilot year (readers from Cell 5 in scope) ---
tmax_y = 2024
tmax = None   # In practice, keep the full cubes in memory from Cell 5.
              # For a Colab memory-light pilot, rebuild per-year slices:
              # tmax[dates >= '2024-01-01'] etc. Shown explicitly:
dates_2024 = dates[(dates >= np.datetime64("2024-01-01")) &
                   (dates <= np.datetime64("2024-12-31"))]
tmax_2024 = tmax[dates_2024[0] <= dates][np.isin(dates, dates_2024)]
precip_2024 = precip[np.isin(dates, dates_2024)]

res = detect_for_years(
    tmax_2024, precip_2024, dates_2024,
    clim["thr_t"], clim["thr_p"],
    {"mean": clim["clim_t_mean"], "sd": clim["clim_t_sd"]},
    {"mean": clim["clim_p_mean"], "sd": clim["clim_p_sd"]},
    [2024], min_days=C.MIN_EPISODE_DAYS)

met = res[2024]
print("2024 national means: episodes %.1f  event days %.1f  severity %.1f"
      % (met["n_episodes"].mean(), met["episode_days"].mean(),
         met["severity"].mean()))

# --- interactive pilot map (EE tile service, no OSM) --------------------------
import ee

ee.Initialize(project=C.GEE_PROJECT)
nepdist = ee.FeatureCollection(C.NEPAL_DISTRICTS_ASSET)
district_image = nepdist.style(color="222222", fillColor="00000000",
                               width=1.2)


def add_ee_layer(map_object, image, vis_params, name):
    map_id = ee.Image(image).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id["tile_fetcher"].url_format, attr="Google Earth Engine",
        name=name, overlay=True, control=True).add_to(map_object)


def grid_to_image(grid, palette, name, m):
    """Overlay a (ny, nx) metric grid as an EE image, clipped to Nepal."""
    img = (ee.Image.pixelCoordinates(ee.Projection("EPSG:4326"))  # placeholder
           )  # replaced below by the direct GeoTIFF overlay path
    # Simpler and exact: display the exported GeoTIFF with folium's
    # raster overlay (small file) instead of reconstructing it in EE:
    import rasterio
    from rasterio.warp import transform_bounds
    from folium.raster_layers import ImageOverlay

    def show(grid_, title_, cmap_):
        fig, ax = plt.subplots(figsize=(4, 4), constrained_layout=True)
        im = ax.imshow(grid_, cmap=cmap_)
        fig.colorbar(im, ax=ax, shrink=0.7)
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=110)
        plt.close(fig)
        buf.seek(0)
        b = rasterio.open(DATA / f"chd_pr_2024.tif")  # georeference template
        wb = transform_bounds(b.crs, "EPSG:4326", *b.bounds)
        ImageOverlay(image=plt.imread(buf), bounds=[[wb[1], wb[0]],
                      [wb[3], wb[2]]], name=title_, opacity=0.75).add_to(m)

    show(grid, name, palette)


import io

m = folium.Map(location=[28.3, 84.1], zoom_start=7, tiles=None,
               control_scale=True)
add_ee_layer(m, district_image, {}, "Nepal districts")
grid_to_image(met["n_episodes"].astype(float), "YlOrRd",
              "2024 compound episodes (count)", m)
grid_to_image(met["max_duration"].astype(float), "PuRd",
              "2024 max episode duration (days)", m)
m.fit_bounds([[26.3, 80.0], [30.5, 88.3]])
folium.LayerControl(collapsed=False).add_to(m)
display(m)

# --- district means (area-weighted in Cell 7; simple mean here for the pilot)
np.savez_compressed(OUT / "pilot_2024_metrics.npz", **met)
np.save(OUT / "pilot_2024_dates.npy", dates_2024)
print("Pilot metrics saved to", OUT / "pilot_2024_metrics.npz")


## Cell 7 — Historical loop 1981–2025 + district aggregation `[run in Colab]`
Per-year checkpointing (one NPZ per year), cos(latitude)-weighted district aggregation, district/province/national CSVs.


In [ ]:
# ============================================================================
# CELL 7 -- HISTORICAL LOOP 1981-2025 + district aggregation (Phases D/E prep)
# ============================================================================
# Runs the same validated detector for every year with checkpointing (one NPZ
# per year so a crash never loses work), then aggregates to districts with
# exact per-pixel area weights (cos(latitude)) -- NOT naive means, because
# equal-angle pixels shrink with latitude.
# [NOT executed in this sandbox: run in Colab after Cell 5.]
# ============================================================================

import json
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

sys.path.insert(0, "/content/chd_pipeline")
import config as C
from local_analysis.run_events import detect_for_years

OUT = Path("/content/chd_outputs")
clim = np.load(OUT / "climatology.npz")
lat = clim["lat"]; lon = clim["lon"]
years = list(range(1981, 2026))

# exact area weights per grid row (fraction of total)
w = np.cos(np.deg2rad(lat))[:, None] * np.ones((1, len(lon)))
w /= w.sum()

# districts shapefile (uploaded District.shp -> uploaded to Colab, or the
# GeoJSON packaged in this repo under data/)
gdf = gpd.read_file("/content/District.shp")     # 77 districts, EPSG:4326

def district_summary(year, met, gdf, lat, lon, w):
    """Aggregate annual metric grids to district means (area-weighted).

    Uses rasterio.features.geometry_mask per district on the shared grid.
    """
    from rasterio.features import geometry_mask
    transform = rasterio.transform.from_bounds(
        lon.min() - 0.05, lat.min() - 0.05, lon.max() + 0.05,
        lat.max() + 0.05, len(lon), len(lat))
    rows = []
    for _, r in gdf.iterrows():
        mask = geometry_mask([r.geometry], out_shape=(len(lat), len(lon)),
                             transform=transform, invert=True)
        wm = w * mask
        wm /= wm.sum()
        rows.append({
            "district": r["DISTRICT"], "province": r["PR_NAME"], "year": year,
            "n_episodes": float((met["n_episodes"] * wm).sum()),
            "episode_days": float((met["episode_days"] * wm).sum()),
            "compound_days": float((met["n_compound_days"] * wm).sum()),
            "mean_duration": float((met["mean_duration"] * wm).sum()),
            "max_duration": float((met["max_duration"] * wm).sum()),
            "severity": float((met["severity"] * wm).sum()),
        })
    return rows

import rasterio
from rasterio.transform import from_bounds as _fb

all_rows = []
for y in years:
    ckpt = OUT / f"metrics_{y}.npz"
    if ckpt.exists():
        met = dict(np.load(ckpt))
    else:
        idx = np.isin(dates, np.arange(np.datetime64(f"{y}-01-01"),
                                       np.datetime64(f"{y + 1}-01-01"),
                                       dtype="datetime64[D]"))
        res = detect_for_years(
            tmax[idx], precip[idx], dates[idx],
            clim["thr_t"], clim["thr_p"],
            {"mean": clim["clim_t_mean"], "sd": clim["clim_t_sd"]},
            {"mean": clim["clim_p_mean"], "sd": clim["clim_p_sd"]},
            [y], min_days=C.MIN_EPISODE_DAYS)
        met = res[y]
        np.savez_compressed(ckpt, **met)
    all_rows += district_summary(y, met, gdf, lat, lon, w)
    print(y, "done")

df = pd.DataFrame(all_rows)
df.to_csv(OUT / "district_annual_chd_1981_2025.csv", index=False)

# province rollup
prov = (df.groupby(["province", "year"])
        [["n_episodes", "episode_days", "severity"]].mean().reset_index())
prov.to_csv(OUT / "province_annual_chd_1981_2025.csv", index=False)

# national series (area-weighted mean of district values)
nat = (df.groupby("year")
       [["n_episodes", "episode_days", "compound_days", "mean_duration",
         "max_duration", "severity"]].mean().reset_index())
nat.to_csv(OUT / "national_annual_chd_1981_2025.csv", index=False)
print("Wrote district/province/national CSVs to", OUT)


## Cell 8 — Trends: Mann–Kendall, Sen, FDR `[run in Colab]`
District-level trends with Yue–Pilon prewhitening and Benjamini–Hochberg field-significance control.


In [ ]:
# ============================================================================
# CELL 8 -- TREND ANALYSIS (Phase E): Mann-Kendall + Sen + FDR
# ============================================================================
# District-level trends of every CHD metric (1981-2025) with Yue-Pilon
# prewhitening option, field-significance control by Benjamini-Hochberg FDR,
# and the national annual series test. Uses core.trends (unit-tested).
# [NOT executed in this sandbox: run in Colab after Cell 7.]
# ============================================================================

import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "/content/chd_pipeline")
from core.trends import fdr_bh, trend_with_prewhitening

OUT = Path("/content/chd_outputs")
df = pd.read_csv(OUT / "district_annual_chd_1981_2025.csv")

METRICS = ["n_episodes", "episode_days", "compound_days", "mean_duration",
           "max_duration", "severity"]

# ---- national trends ---------------------------------------------------------
nat = pd.read_csv(OUT / "national_annual_chd_1981_2025.csv")
rows = []
for m in METRICS:
    r = trend_with_prewhitening(nat[m].values, prewhiten=True)
    rows.append({"metric": m, "sen_slope_per_decade": r["slope"] * 10,
                 "ci_lo": r["ci_lo"] * 10, "ci_hi": r["ci_hi"] * 10,
                 "p": r["p"], "tau": r["tau"]})
print(pd.DataFrame(rows).to_string(index=False))

# ---- district trends + FDR ---------------------------------------------------
out_rows = []
for (dist, prov), g in df.groupby(["district", "province"]):
    g = g.sort_values("year")
    for m in METRICS:
        r = trend_with_prewhitening(g[m].values, prewhiten=True)
        out_rows.append({"district": dist, "province": prov, "metric": m,
                         "slope_decade": r["slope"] * 10, "p": r["p"],
                         "tau": r["tau"]})
tr = pd.DataFrame(out_rows)

# FDR per metric across the 77 districts
tr["p_fdr"] = np.nan
for m, g in tr.groupby("metric"):
    tr.loc[g.index, "p_fdr"] = g["p"].values  # placeholder for mapping
sig_masks = {}
for m, g in tr.groupby("metric"):
    sig_masks[m] = fdr_bh(g["p"].values, alpha=0.05)
    tr.loc[g.index, "fdr_sig"] = sig_masks[m]

tr.to_csv(OUT / "district_trends_chd.csv", index=False)
print("\nSignificant (FDR 5%) district trends per metric:")
print(tr.groupby("metric")["fdr_sig"].sum().to_string())


## Cell 9 — MODIS vegetation response (Phase F) `[run in Colab]`
MOD13Q1 v061, SummaryQA ≤ 1 filter, scale 1e-4, district-monthly table exports in yearly chunks.


In [ ]:
# ============================================================================
# CELL 9 -- MODIS NDVI/EVI vegetation response (Phase F)
# ============================================================================
# MOD13Q1 (Collection 6.1): NDVI/EVI scale 1e-4; quality filter SummaryQA<=1
# (bits 0-1: 0=good, 1=marginal; excludes cloud/snow fills). District-monthly
# means are exported as a GEE table in year chunks (the audited two-stage
# pattern), then matched to CHD exposure months in Python.
# [NOT executed in this sandbox: run in Colab.]
# ============================================================================

import ee

ee.Initialize(project="mygeepr")
nepdist = ee.FeatureCollection("projects/mygeepr/assets/Nepaldist")

mod13 = (ee.ImageCollection("MODIS/061/MOD13Q1")
         .select(["NDVI", "EVI", "SummaryQA"]))


def qa_mask(im):
    qa = im.select("SummaryQA")
    ok = qa.bitwiseAnd(3).lte(1)          # good + marginal quality
    vi = im.select(["NDVI", "EVI"]).multiply(0.0001)
    return vi.updateMask(ok).copyProperties(im, ["system:time_start"])


mod13q = mod13.map(qa_mask)


def year_table(year):
    ic = mod13q.filterDate(f"{year}-01-01", f"{year + 1}-01-01")
    fc = ic.map(lambda im: im.reduceRegions(
        collection=nepdist,
        reducer=ee.Reducer.mean().combine(
            ee.Reducer.stdDev(), sharedInputs=True),
        scale=250)).flatten() \
        .select(["DISTRICT", "PR_NAME", "mean", "mean_1", "system:index"])
    task = ee.batch.Export.table.toDrive(
        collection=fc, description=f"CHD_NDVI_{year}",
        folder="nepal_chd_cubes", fileFormat="CSV",
        selectors=["DISTRICT", "PR_NAME", "mean", "mean_1", "system:index"])
    task.start()
    return task.id


ids = [year_table(y) for y in range(2001, 2026)]
print("Started", len(ids), "NDVI table exports (2001-2025).")
# Each yearly table has 23 16-day composites x 77 districts ~ 1.8k rows.
# After completion, merge in pandas; composite month = from system:index
# (e.g. 2024-01-01 -> month 1). Then compute per-district growing-season
# anomalies (z vs 2001-2020) and composite with the CHD classes from Cells 6-7:
# class means (normal / heat-only / dry-only / compound) and lag responses
# (composite of the NEXT 16-day period for delayed response).


## Cell 10 — NEX-GDDP-CMIP6 projections (Phase H) `[run in Colab]`
S3 layout verified from the live bucket; QDM bias adjustment; observed 1991–2020 thresholds applied to bias-adjusted future series. Run ONE model first; the full ensemble needs planning.


In [ ]:
# ============================================================================
# CELL 10 -- NEX-GDDP-CMIP6 future projections (Phase H)
# ============================================================================
# Access layout VERIFIED against the public S3 bucket (2026-11):
#   s3://nex-gddp-cmip6/NEX-GDDP-CMIP6/{model}/{scenario}/{realization}/
#       {variable}/{var}_day_{model}_{scenario}_{ri}_gn_{year}[_vX.X].nc
#   scenarios: historical, ssp126, ssp245, ssp370, ssp585; variables incl.
#   tasmax, pr (plus hurs etc.). File variants: base, _v1.1, _v2.0.
# Units: tasmax K -> degC; pr kg m-2 s-1 -> mm/day (x86400).
# Bias adjustment: Quantile Delta Mapping (Cannon et al. 2015) against the
#   1991-2020 observed climatology on the SAME grid (quantiles computed per
#   calendar-day window). CHD detection then reuses the SAME observed
#   1991-2020 thresholds (thr_t, thr_p) applied to bias-adjusted series --
#   the consistent-threshold requirement of the concept note (Phase H).
# [NOT executed in this sandbox: run in Colab -- needs ~50 GB working disk
#  for the full ensemble; start with ONE model for the pilot.]
# ============================================================================

import sys
from pathlib import Path

import numpy as np
import s3fs
import xarray as xr

sys.path.insert(0, "/content/chd_pipeline")
import config as C
from core.events import calendar_thresholds, classify_days, rolling_sum

fs = s3fs.S3FileSystem(anon=True)
ROOT = f"{C.GDDP_S3}/{C.GDDP_ROOT}"


def gddp_files(model, scenario, var_gddp, years):
    """Enumerate year files for one model/scenario; tolerate version suffixes
    (base -> _v2.0 -> _v1.1 priority, verified naming)."""
    out = []
    ri = C.GDDP_REALIZATION
    for y in years:
        base = f"{C.GDDP_ROOT}/{model}/{scenario}/{ri}/{var_gddp}/" \
               f"{var_gddp}_day_{model}_{scenario}_{ri}_gn_{y}"
        cands = [base + ".nc", base + "_v2.0.nc", base + "_v1.1.nc"]
        hit = next((f"s3://{C.GDDP_S3.split('//')[1]}/{c}"
                    for c in cands if fs.exists(f"{C.GDDP_ROOT}/{model}/{scenario}/{ri}/{var_gddp}/" + c.split('/')[-1])), None)
        if hit is None:
            print("MISSING:", model, scenario, var_gddp, y)
        else:
            out.append(hit)
    return out


def load_subset(model, scenario, var_gddp, years, bbox):
    """Open the year files and subset to the Nepal bbox (lazy, then load)."""
    files = gddp_files(model, scenario, var_gddp, years)
    ds = xr.open_mfdataset(files, combine="by_coords", engine="h5netcdf",
                           chunks={"time": 366})
    sub = ds[var_gddp].sel(lat=slice(bbox[3], bbox[1]),
                           lon=slice(bbox[0], bbox[2])).load()
    if var_gddp == "tasmax":
        sub = sub - 273.15
    elif var_gddp == "pr":
        sub = sub * 86400.0
    return sub


def quantile_delta_mapping(obs_train, mod_train, mod_future):
    """1-D QDM per grid cell over the full daily distribution.

    obs_train, mod_train : (Tb, ...) base-period observed / modeled arrays
    mod_future : (Tf, ...) future modeled series
    Corrects model bias and preserves the model's quantile change signal.
    """
    Tf = mod_future.shape[0]
    qs = np.linspace(0.005, 0.995, 199)
    qo = np.nanquantile(obs_train, qs, axis=0)
    qmh = np.nanquantile(mod_train, qs, axis=0)
    qmf = np.nanquantile(mod_future, qs, axis=0)
    # delta change factor per quantile (additive for temperature)
    delta = qmf - qmh
    # bias correction of the future series quantile by quantile
    out = np.empty_like(mod_future, dtype=np.float32)
    # map each future day's value through the model's historical CDF, then
    # add the corresponding delta. Vectorized with searchsorted per cell.
    flat = mod_future.reshape(Tf, -1)
    for c in range(flat.shape[1]):
        col = flat[:, c]
        pos = np.searchsorted(qmh[:, c], col)
        pos = np.clip(pos, 0, len(qs) - 1)
        out[:, c] = col + delta[pos, c]
    return out.reshape(mod_future.shape)


def projected_chd_metrics(model, scenario, years, thr_t, thr_p,
                          window_days=30):
    """Bias-adjusted CHD metrics for a future period, using the OBSERVED
    1991-2020 thresholds (consistent baseline across past and future)."""
    hist_y = list(range(1991, 2021))
    obs = np.load("/content/chd_outputs/climatology.npz")
    # observed base period arrays from the cubes (loaded in Cell 5 scope)
    obs_t = tmax[base_mask]; obs_p = p30[base_mask & np.isfinite(p30)]

    fut_t = load_subset(model, scenario, "tasmax", years, C.BBOX).values
    fut_p = load_subset(model, scenario, "pr", years, C.BBOX).values
    fut_p30 = rolling_sum(fut_p, window_days)

    mod_hist_t = load_subset(model, "historical", "tasmax",
                             hist_y, C.BBOX).values
    mod_hist_p30 = rolling_sum(
        load_subset(model, "historical", "pr", hist_y, C.BBOX).values,
        window_days)

    adj_t = quantile_delta_mapping(obs_t, mod_hist_t, fut_t)
    adj_p = quantile_delta_mapping(obs_p, mod_hist_p30, fut_p30)

    fut_dates = np.arange(np.datetime64(f"{years[0]}-01-01"),
                          np.datetime64(f"{years[-1] + 1}-01-01"))
    from local_analysis.run_events import detect_for_years
    res = detect_for_years(adj_t, adj_p, fut_dates,
                           thr_t, thr_p,
                           {"mean": obs["clim_t_mean"],
                            "sd": obs["clim_t_sd"]},
                           {"mean": obs["clim_p_mean"],
                            "sd": obs["clim_p_sd"]},
                           years, min_days=C.MIN_EPISODE_DAYS)
    return {y: {k: float(np.nanmean(v)) for k, v in res[y].items()}
            for y in years}


## Roadmap after the historical dataset passes review

1. **Phase F** — vegetation anomalies under the four day-classes (Cell 9 output × Cell 6/7 classes), immediate + 1-composite lag.
2. **Phase G** — merge MoALD district crop statistics, build the panel (`core/panel.py`), report associations with clustered SE.
3. **Phase H/I** — 5-model ensemble for SSP2-4.5/SSP5-8.5, QDM adjusted, three projection windows; exposure hotspots vs WorldCover cropland (class 40).
4. Produce Figures 3–10 with `viz/figures.py` on REAL arrays only.
